In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns 

In [3]:
# Load dataset
df = pd.read_csv("Unicorn Startups.csv")

# Quick first look
print(df.shape)
df.head()

(99, 8)


,STARTUP NAME,INDUSTRY,FOUNDING YEAR,UNICORN ENTRY YEAR,PROFIT/LOSS FY22,CURRENT VALUATION,ACQUISITIONS,STATUS
0,Perfios,SaaS (Fintech),2008,2024,$0.94 Million,$1 Billion,3.0,Private
1,Krutrim,Research Services (AI),2023,2024,NaN,$1 Billion,NaN,Private
2,Zepto,Quick Commerce,2021,2023,-$47.1 Million,$1.4 Billion,NaN,Private
3,Molbio Diagnostics,HealthTech (MedTech),2010,2022,NaN,$1.5 Billion,1.0,Private
4,Tata 1mg,HealthTech,2015,2022,-$65 Million,$1.25 Billion,NaN,Acquired


In [11]:
print("\n--- Missing Values ---")
print(df.isnull().sum())

for col in df.columns:
    print(f"\n {col}")

print("\n--- Sample Values ---")
for col in df.columns:
    print(f"\n{col}: {df[col].unique()[:5]}")


--- Missing Values ---
STARTUP NAME           0
INDUSTRY               0
FOUNDING YEAR          0
UNICORN ENTRY YEAR     0
PROFIT/LOSS FY22      27
CURRENT VALUATION      1
ACQUISITIONS           6
STATUS                 0
dtype: int64

 STARTUP NAME

 INDUSTRY

 FOUNDING YEAR

 UNICORN ENTRY YEAR

 PROFIT/LOSS FY22

 CURRENT VALUATION

 ACQUISITIONS

 STATUS 

--- Sample Values ---

STARTUP NAME: ['Perfios' 'Krutrim' 'Zepto' 'Molbio Diagnostics' 'Tata 1mg']

INDUSTRY: ['SaaS (Fintech)' 'Research Services (AI)' 'Quick Commerce'
 'HealthTech (MedTech)' 'HealthTech']

FOUNDING YEAR: [2008 2023 2021 2010 2015]

UNICORN ENTRY YEAR: [2024 2023 2022 2021 2020]

PROFIT/LOSS FY22: ['$0.94 Million' nan '-$47.1 Million' '-$65 Million' '-$11.67 Million']

CURRENT VALUATION: ['$1 Billion' '$1.4 Billion' '$1.5 Billion' '$1.25 Billion' '$1.3 Billion']

ACQUISITIONS: [ 3. nan  1.  5.  0.]

STATUS : ['Private ' 'Acquired ' 'Listed ' 'Acquired by Flipkart ' 'IPO-Bound ']


In [12]:
# Standardizing column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('/', '_')
print(df.columns.tolist())

['startup_name', 'industry', 'founding_year', 'unicorn_entry_year', 'profit_loss_fy22', 'current_valuation', 'acquisitions', 'status']


In [13]:
def clean_valuation(val):
    if pd.isna(val) or val == 'NA':
        return np.nan
    val = str(val).replace('$', '').replace(',', '').strip()
    if 'Billion' in val:
        return float(val.replace('Billion', '').strip()) * 1000  # convert to millions
    elif 'Million' in val:
        return float(val.replace('Million', '').strip())
    else:
        return np.nan

df['valuation_million_usd'] = df['current_valuation'].apply(clean_valuation)
print(df[['startup_name', 'current_valuation', 'valuation_million_usd']].head(10))

         startup_name current_valuation  valuation_million_usd
0             Perfios        $1 Billion                 1000.0
1             Krutrim        $1 Billion                 1000.0
2               Zepto      $1.4 Billion                 1400.0
3  Molbio Diagnostics      $1.5 Billion                 1500.0
4            Tata 1mg     $1.25 Billion                 1250.0
5          Shiprocket      $1.3 Billion                 1300.0
6                5ire      $1.5 Billion                 1500.0
7             OneCard      $1.4 Billion                 1400.0
8       PhysicsWallah      $1.1 Billion                 1100.0
9         LeadSquared        $1 Billion                 1000.0


In [15]:
#Clean PROFIT/LOSS
def clean_profit_loss(val):
    if pd.isna(val) or str(val).strip() == 'NA':
        return np.nan
    val = str(val).replace('$', '').replace(',', '').strip()
    # Handle the Groww anomaly (missing 'Million')
    if 'Million' not in val and 'Billion' not in val:
        try:
            return float(val)  # already a number like -29.5
        except:
            return np.nan
    if 'Billion' in val:
        return float(val.replace('Billion', '').strip()) * 1000
    elif 'Million' in val:
        return float(val.replace('Million', '').strip())
    return np.nan

df['profit_loss_million_usd'] = df['profit_loss_fy22'].apply(clean_profit_loss)
print(df[['startup_name', 'profit_loss_fy22', 'profit_loss_million_usd']].head(10))

         startup_name profit_loss_fy22  profit_loss_million_usd
0             Perfios    $0.94 Million                     0.94
1             Krutrim              NaN                      NaN
2               Zepto   -$47.1 Million                   -47.10
3  Molbio Diagnostics              NaN                      NaN
4            Tata 1mg     -$65 Million                   -65.00
5          Shiprocket  -$11.67 Million                   -11.67
6                5ire              NaN                      NaN
7             OneCard   -$22.8 Million                   -22.80
8       PhysicsWallah    $16.4 Million                    16.40
9         LeadSquared    -$7.7 Million                    -7.70


In [16]:
#Standardise INDUSTRY
def standardise_industry(ind):
    ind = str(ind).strip()
    if any(x in ind for x in ['Fintech', 'Financial Technology', 'Stockbroker', 'Financial Services', 'Insurance', 'Insurtech', 'Cryptocurren', 'Blockchain']):
        return 'Fintech & Financial Services'
    elif any(x in ind for x in ['Edtech', 'Education']):
        return 'Edtech'
    elif any(x in ind for x in ['SaaS', 'Saas', 'Cloud', 'Software', 'Enterprisetech']):
        return 'SaaS & Enterprise Tech'
    elif any(x in ind for x in ['E-commerce', 'Ecommerce', 'B2B E-commerce', 'B2C E-commerce']):
        return 'E-commerce'
    elif any(x in ind for x in ['Health', 'MedTech', 'Pharma', 'Diagnostics']):
        return 'Healthtech'
    elif any(x in ind for x in ['Logistics', 'Supply Chain']):
        return 'Logistics'
    elif any(x in ind for x in ['Media', 'Entertainment', 'Gaming', 'ESports']):
        return 'Media & Entertainment'
    elif any(x in ind for x in ['Automotive', 'Auto']):
        return 'Automotive'
    elif any(x in ind for x in ['PropTech', 'Real Estate']):
        return 'PropTech'
    elif 'Manufacturing' in ind:
        return 'Manufacturing'
    elif 'Travel' in ind or 'Hospitality' in ind:
        return 'Travel & Hospitality'
    elif 'Research' in ind or 'AI' in ind:
        return 'AI & Research'
    else:
        return 'Other'

df['industry_clean'] = df['industry'].apply(standardise_industry)
print(df['industry_clean'].value_counts())

industry_clean
Fintech & Financial Services    26
E-commerce                      24
SaaS & Enterprise Tech          14
Other                            7
Healthtech                       7
Edtech                           6
Media & Entertainment            6
Logistics                        3
Automotive                       2
AI & Research                    1
PropTech                         1
Manufacturing                    1
Travel & Hospitality             1
Name: count, dtype: int64


In [17]:
#Standardise STATUS & ACQUISITIONS
# Normalise status
def standardise_status(s):
    s = str(s).strip()
    if 'Acquired' in s:
        return 'Acquired'
    return s

df['status_clean'] = df['status'].apply(standardise_status)

# Convert acquisitions to numeric
df['acquisitions'] = pd.to_numeric(df['acquisitions'], errors='coerce')

print(df['status_clean'].value_counts())

status_clean
Private      89
Acquired      5
Listed        4
IPO-Bound     1
Name: count, dtype: int64


In [18]:
#Adding calculations
# Years to become a unicorn
df['years_to_unicorn'] = df['unicorn_entry_year'] - df['founding_year']

# Profitability flag
df['is_profitable'] = df['profit_loss_million_usd'].apply(
    lambda x: 'Profitable' if pd.notna(x) and x > 0 else ('Loss-Making' if pd.notna(x) and x < 0 else 'Unknown')
)

print(df[['startup_name', 'founding_year', 'unicorn_entry_year', 'years_to_unicorn', 'is_profitable']].head(10))

         startup_name  founding_year  unicorn_entry_year  years_to_unicorn  \
0             Perfios           2008                2024                16   
1             Krutrim           2023                2024                 1   
2               Zepto           2021                2023                 2   
3  Molbio Diagnostics           2010                2022                12   
4            Tata 1mg           2015                2022                 7   
5          Shiprocket           2017                2022                 5   
6                5ire           2021                2022                 1   
7             OneCard           2018                2022                 4   
8       PhysicsWallah           2016                2022                 6   
9         LeadSquared           2011                2022                11   

  is_profitable  
0    Profitable  
1       Unknown  
2   Loss-Making  
3       Unknown  
4   Loss-Making  
5   Loss-Making  
6       Unknown

In [19]:
df.to_csv('Unicorn_Startups_Cleaned.csv', index=False)
print("✅ Clean dataset saved!")
print(f"Shape: {df.shape}")
df.info()

✅ Clean dataset saved!
Shape: (99, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   startup_name             99 non-null     object 
 1   industry                 99 non-null     object 
 2   founding_year            99 non-null     int64  
 3   unicorn_entry_year       99 non-null     int64  
 4   profit_loss_fy22         72 non-null     object 
 5   current_valuation        98 non-null     object 
 6   acquisitions             93 non-null     float64
 7   status                   99 non-null     object 
 8   valuation_million_usd    98 non-null     float64
 9   profit_loss_million_usd  71 non-null     float64
 10  industry_clean           99 non-null     object 
 11  status_clean             99 non-null     object 
 12  years_to_unicorn         99 non-null     int64  
 13  is_profitable            99 non-null     ob